# Lọc mấy tấm hình lấy ra train dựa theo dữ liệu đã filter 5 core
File df_inter.label.parquet, df_inter.parquet lấy ở bước 1 hoặc 2 điều được

In [3]:
import os
import pandas as pd

In [4]:
PATH = "../data/2023/"

In [10]:
df = pd.read_parquet(os.path.join(PATH, "df_meta.parquet"))

In [11]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 35997 entries, 0 to 35996
Data columns (total 16 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   main_category    35236 non-null  str    
 1   title            35997 non-null  str    
 2   average_rating   35997 non-null  float64
 3   rating_number    35997 non-null  int64  
 4   features         35997 non-null  object 
 5   description      35997 non-null  object 
 6   price            17404 non-null  float64
 7   images           35997 non-null  object 
 8   videos           35997 non-null  object 
 9   store            35849 non-null  str    
 10  categories       35997 non-null  object 
 11  details          35997 non-null  object 
 12  asin             35997 non-null  str    
 13  bought_together  0 non-null      float64
 14  subtitle         0 non-null      float64
 15  author           0 non-null      object 
dtypes: float64(4), int64(1), object(7), str(4)
memory usage: 9.1+ MB


In [9]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 35997 entries, 0 to 35996
Data columns (total 19 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   itemID           35997 non-null  int64  
 1   asin             35997 non-null  str    
 2   main_category    35997 non-null  str    
 3   title            35997 non-null  str    
 4   average_rating   35997 non-null  float64
 5   rating_number    35997 non-null  int64  
 6   features         35997 non-null  object 
 7   description      35997 non-null  object 
 8   price            17404 non-null  float64
 9   images           35997 non-null  object 
 10  videos           35997 non-null  object 
 11  store            35997 non-null  str    
 12  categories       35997 non-null  object 
 13  details          35997 non-null  object 
 14  bought_together  0 non-null      float64
 15  subtitle         0 non-null      float64
 16  author           0 non-null      object 
 17  combined_text    35997 

In [7]:
df.head()

,userID,itemID,rating,timestamp,reviewerID,asin,x_label
0,0,4,4,1655859477958,AEO4M665ZOCBF7HEFRMTUDHLSB5Q,B004JU0H6O,0
1,0,3,5,1655860079499,AEO4M665ZOCBF7HEFRMTUDHLSB5Q,B082WJTFRR,0
2,0,2,4,1655860597696,AEO4M665ZOCBF7HEFRMTUDHLSB5Q,B08FZJ3YHH,0
3,0,1,5,1655862402295,AEO4M665ZOCBF7HEFRMTUDHLSB5Q,B017IQZ9OK,1
4,0,0,3,1657839829629,AEO4M665ZOCBF7HEFRMTUDHLSB5Q,B086QM7FVT,2


In [40]:
df_5core = pd.read_csv("ratings_Baby_5core.csv", names=['userID', 'itemID', 'rating', 'timestamp'], header=None)

In [43]:
import polars as pl
import pandas as pd

# Get unique ASINs from the review data (Polars LazyFrame)
lazy_review_asins = lazy_df_review_2023.select(pl.col('asin')).unique()

# Get unique itemIDs from df_5core (Pandas DataFrame) and convert to Polars DataFrame
df_5core_itemids_pl = pl.DataFrame({'itemID': df_5core['itemID'].unique()})

# DF1: asin from lazy_df_review_2023 that exist in df_5core['itemID']
df_review_asins_in_5core = (
    lazy_review_asins.join(
        df_5core_itemids_pl.rename({'itemID': 'asin'}).lazy(), # Convert to LazyFrame for join
        on='asin',
        how='inner'
    )
    .collect()
)

print("ASINs from reviews also found in 5-core itemIDs:")
display(df_review_asins_in_5core)

ASINs from reviews also found in 5-core itemIDs:


asin
str
"""B002OOWADU"""
"""B004YU23IY"""
"""B000SXSMYK"""
"""B0032BEFPK"""
"""B004071ZO4"""
…
"""B004A9ZF3Y"""
"""B004CLYZ0O"""
"""B00DK9C3IA"""


In [44]:
# DF2: df_5core['itemID'] not existing in lazy_df_review_2023['asin']
df_5core_itemids_not_in_review = (
    df_5core_itemids_pl.lazy().join( # Convert to LazyFrame for join
        lazy_review_asins.rename({'asin': 'itemID'}).with_columns(pl.lit(True).alias('in_review')),
        on='itemID',
        how='left'
    )
    .filter(pl.col('in_review').is_null())
    .select(pl.col('itemID'))
    .collect()
)

print("ItemIDs from 5-core not found in review ASINs:")
display(df_5core_itemids_not_in_review)

ItemIDs from 5-core not found in review ASINs:


itemID
str
"""B00000IZQI"""
"""B00000J3LL"""
"""B00002JXFI"""
"""B00004C8S8"""
"""B00004TFLB"""
…
"""B00IVNEGLW"""
"""B00IVNEGOO"""
"""B00J8B47ME"""


In [46]:
4353+2697

7050

In [49]:
df_5core["itemID"].drop_duplicates().count()

np.int64(7050)

In [52]:
import polars as pl
import pandas as pd

# Get unique parent_asins from the meta data (Polars LazyFrame)
lazy_meta_parent_asins = lazy_df_meta_2023.select(pl.col('parent_asin')).unique()

# Get unique itemIDs from df_5core (Pandas DataFrame) and convert to Polars DataFrame
df_5core_itemids_pl = pl.DataFrame({'itemID': df_5core['itemID'].unique()})

# DF1: parent_asin from lazy_df_meta_2023 that exist in df_5core['itemID']
df_meta_parent_asins_in_5core = (
    lazy_meta_parent_asins.join(
        df_5core_itemids_pl.rename({'itemID': 'parent_asin'}).lazy(), # Convert to LazyFrame for join
        on='parent_asin',
        how='inner'
    )
    .collect()
)

print("Parent ASINs from meta also found in 5-core itemIDs:")
display(df_meta_parent_asins_in_5core)

Parent ASINs from meta also found in 5-core itemIDs:


parent_asin
str
"""B00AAIA8PE"""
"""B00003TL7P"""
"""B003B09V2Y"""
"""B001QKJ9SC"""
"""B000R9YXWA"""
…
"""B002LZX2BG"""
"""B000T9792O"""
"""B00318CLNM"""


In [54]:
# Extract parent_asin values from df_meta_parent_asins_in_5core
parent_asins_in_5core_list = df_meta_parent_asins_in_5core['parent_asin'].to_list()

# Filter lazy_df_meta_2023 to get full columns for these parent_asins
df_meta_full_columns_in_5core = (
    lazy_df_meta_2023
    .filter(pl.col('parent_asin').is_in(parent_asins_in_5core_list))
    .collect()
)

print("Filtered metadata with parent_asins found in 5-core (first 5 rows):")
display(df_meta_full_columns_in_5core.head())

Filtered metadata with parent_asins found in 5-core (first 5 rows):


shape: (5, 17)
┌────────────┬───────────┬───────────┬───────────┬───┬───────────┬──────────┬───────────┬──────────┐
│ main_categ ┆ title     ┆ average_r ┆ rating_nu ┆ … ┆ bought_to ┆ subtitle ┆ author    ┆ df_index │
│ ory        ┆ ---       ┆ ating     ┆ mber      ┆   ┆ gether    ┆ ---      ┆ ---       ┆ ---      │
│ ---        ┆ str       ┆ ---       ┆ ---       ┆   ┆ ---       ┆ f64      ┆ struct[3] ┆ i64      │
│ str        ┆           ┆ f64       ┆ i64       ┆   ┆ f64       ┆          ┆           ┆          │
╞════════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪══════════╪═══════════╪══════════╡
│ Baby       ┆ Nuby      ┆ 4.4       ┆ 52        ┆ … ┆ null      ┆ null     ┆ null      ┆ 3        │
│            ┆ iMonster  ┆           ┆           ┆   ┆           ┆          ┆           ┆          │
│            ┆ Toddler   ┆           ┆           ┆   ┆           ┆          ┆           ┆          │
│            ┆ Bowl      ┆           ┆           ┆   ┆           ┆          ┆           ┆          │
│ Baby       ┆ BabyKicks ┆ 4.6       ┆ 243       ┆ … ┆ null      ┆ null     ┆ null      ┆ 100      │
│            ┆ Natural & ┆           ┆           ┆   ┆           ┆          ┆           ┆          │
│            ┆ Organic   ┆           ┆           ┆   ┆           ┆          ┆           ┆          │
│            ┆ 10…       ┆           ┆           ┆   ┆           ┆          ┆           ┆          │
│ Baby       ┆ Hands     ┆ 3.3       ┆ 41        ┆ … ┆ null      ┆ null     ┆ null      ┆ 113      │
│            ┆ Free Brea ┆           ┆           ┆   ┆           ┆          ┆           ┆          │
│            ┆ stpump    ┆           ┆           ┆   ┆           ┆          ┆           ┆          │
│            ┆ Bra Halt… ┆           ┆           ┆   ┆           ┆          ┆           ┆          │
│ Tools &    ┆ Cloud B   ┆ 3.7       ┆ 54        ┆ … ┆ null      ┆ null     ┆ null      ┆ 143      │
│ Home Impro ┆ Twilight  ┆           ┆           ┆   ┆           ┆          ┆           ┆          │
│ vement     ┆ Ladybug   ┆           ┆           ┆   ┆           ┆          ┆           ┆          │
│            ┆ Green     ┆           ┆           ┆   ┆           ┆          ┆           ┆          │
│ Baby       ┆ Disney    ┆ 3.9       ┆ 119       ┆ … ┆ null      ┆ null     ┆ null      ┆ 316      │
│            ┆ Cars      ┆           ┆           ┆   ┆           ┆          ┆           ┆          │
│            ┆ Folding   ┆           ┆           ┆   ┆           ┆          ┆           ┆          │
│            ┆ Potty     ┆           ┆           ┆   ┆           ┆          ┆           ┆          │
│            ┆ Seat…     ┆           ┆           ┆   ┆           ┆          ┆           ┆          │
└────────────┴───────────┴───────────┴───────────┴───┴───────────┴──────────┴───────────┴──────────┘

In [55]:
# Save the resulting DataFrame to a Parquet file
output_parquet_path = "filtered_meta_with_5core_asins.parquet"
df_meta_full_columns_in_5core.write_parquet(output_parquet_path)

print(f"DataFrame saved to {output_parquet_path}")

DataFrame saved to filtered_meta_with_5core_asins.parquet


In [58]:
# Optionally, copy to Google Drive
!cp "filtered_meta_with_5core_asins.parquet" "{PATH}/2014/filtered_meta_with_5core_asins.parquet"

In [59]:
df_meta_full_columns_in_5core

shape: (3_443, 17)
┌────────────┬───────────┬───────────┬───────────┬───┬───────────┬──────────┬───────────┬──────────┐
│ main_categ ┆ title     ┆ average_r ┆ rating_nu ┆ … ┆ bought_to ┆ subtitle ┆ author    ┆ df_index │
│ ory        ┆ ---       ┆ ating     ┆ mber      ┆   ┆ gether    ┆ ---      ┆ ---       ┆ ---      │
│ ---        ┆ str       ┆ ---       ┆ ---       ┆   ┆ ---       ┆ f64      ┆ struct[3] ┆ i64      │
│ str        ┆           ┆ f64       ┆ i64       ┆   ┆ f64       ┆          ┆           ┆          │
╞════════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪══════════╪═══════════╪══════════╡
│ Baby       ┆ Nuby      ┆ 4.4       ┆ 52        ┆ … ┆ null      ┆ null     ┆ null      ┆ 3        │
│            ┆ iMonster  ┆           ┆           ┆   ┆           ┆          ┆           ┆          │
│            ┆ Toddler   ┆           ┆           ┆   ┆           ┆          ┆           ┆          │
│            ┆ Bowl      ┆           ┆           ┆   ┆           ┆          ┆           ┆          │
│ Baby       ┆ BabyKicks ┆ 4.6       ┆ 243       ┆ … ┆ null      ┆ null     ┆ null      ┆ 100      │
│            ┆ Natural & ┆           ┆           ┆   ┆           ┆          ┆           ┆          │
│            ┆ Organic   ┆           ┆           ┆   ┆           ┆          ┆           ┆          │
│            ┆ 10…       ┆           ┆           ┆   ┆           ┆          ┆           ┆          │
│ Baby       ┆ Hands     ┆ 3.3       ┆ 41        ┆ … ┆ null      ┆ null     ┆ null      ┆ 113      │
│            ┆ Free Brea ┆           ┆           ┆   ┆           ┆          ┆           ┆          │
│            ┆ stpump    ┆           ┆           ┆   ┆           ┆          ┆           ┆          │
│            ┆ Bra Halt… ┆           ┆           ┆   ┆           ┆          ┆           ┆          │
│ Tools &    ┆ Cloud B   ┆ 3.7       ┆ 54        ┆ … ┆ null      ┆ null     ┆ null      ┆ 143      │
│ Home Impro ┆ Twilight  ┆           ┆           ┆   ┆           ┆          ┆           ┆          │
│ vement     ┆ Ladybug   ┆           ┆           ┆   ┆           ┆          ┆           ┆          │
│            ┆ Green     ┆           ┆           ┆   ┆           ┆          ┆           ┆          │
│ Baby       ┆ Disney    ┆ 3.9       ┆ 119       ┆ … ┆ null      ┆ null     ┆ null      ┆ 316      │
│            ┆ Cars      ┆           ┆           ┆   ┆           ┆          ┆           ┆          │
│            ┆ Folding   ┆           ┆           ┆   ┆           ┆          ┆           ┆          │
│            ┆ Potty     ┆           ┆           ┆   ┆           ┆          ┆           ┆          │
│            ┆ Seat…     ┆           ┆           ┆   ┆           ┆          ┆           ┆          │
│ …          ┆ …         ┆ …         ┆ …         ┆ … ┆ …         ┆ …        ┆ …         ┆ …        │
│ Baby       ┆ Blueberry ┆ 3.1       ┆ 27        ┆ … ┆ null      ┆ null     ┆ null      ┆ 215364   │
│            ┆ Coveralls ┆           ┆           ┆   ┆           ┆          ┆           ┆          │
│            ┆ Hook/Loop ┆           ┆           ┆   ┆           ┆          ┆           ┆          │
│            ┆ …         ┆           ┆           ┆   ┆           ┆          ┆           ┆          │
│ null       ┆ Vital     ┆ 3.0       ┆ 27        ┆ … ┆ null      ┆ null     ┆ null      ┆ 215588   │
│            ┆ Baby      ┆           ┆           ┆   ┆           ┆          ┆           ┆          │
│            ┆ Toddler   ┆           ┆           ┆   ┆           ┆          ┆           ┆          │
│            ┆ Trainer   ┆           ┆           ┆   ┆           ┆          ┆           ┆          │
│            ┆ Cup…      ┆           ┆           ┆   ┆           ┆          ┆           ┆          │
│ null       ┆ Blueberry ┆ 3.0       ┆ 1         ┆ … ┆ null      ┆ null     ┆ null      ┆ 215615   │
│            ┆ Minky     ┆           ┆           ┆   ┆           ┆          ┆           ┆          │
│            ┆ Diaper    ┆           ┆    

In [61]:
df_meta_2014 = pd.read_parquet("filtered_meta_with_5core_asins.parquet")

In [62]:
df_meta_2014.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3443 entries, 0 to 3442
Data columns (total 17 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   main_category    3338 non-null   object 
 1   title            3443 non-null   object 
 2   average_rating   3443 non-null   float64
 3   rating_number    3443 non-null   int64  
 4   features         3443 non-null   object 
 5   description      3443 non-null   object 
 6   price            522 non-null    float64
 7   images           3443 non-null   object 
 8   videos           3443 non-null   object 
 9   store            3427 non-null   object 
 10  categories       3443 non-null   object 
 11  details          3443 non-null   object 
 12  parent_asin      3443 non-null   object 
 13  bought_together  0 non-null      float64
 14  subtitle         0 non-null      float64
 15  author           0 non-null      object 
 16  df_index         3443 non-null   int64  
dtypes: float64(4),

In [53]:
# DF2: df_5core['itemID'] not existing in lazy_df_meta_2023['parent_asin']
df_5core_itemids_not_in_meta = (
    df_5core_itemids_pl.lazy().join( # Convert to LazyFrame for join
        lazy_meta_parent_asins.rename({'parent_asin': 'itemID'}).with_columns(pl.lit(True).alias('in_meta')),
        on='itemID',
        how='left'
    )
    .filter(pl.col('in_meta').is_null())
    .select(pl.col('itemID'))
    .collect()
)

print("ItemIDs from 5-core not found in meta parent ASINs:")
display(df_5core_itemids_not_in_meta)

ItemIDs from 5-core not found in meta parent ASINs:


itemID
str
"""B00000IZQI"""
"""B00000J3LL"""
"""B00002JV9S"""
"""B00002JXFI"""
"""B000046S3W"""
…
"""B00J3TVRSS"""
"""B00J8B47ME"""
"""B00JB2PJ72"""


## Read 2014 dataset

In [113]:
df_filtered_meta_5core_2014 = pd.read_parquet("filtered_meta_with_5core_asins.parquet")

In [114]:
df_filtered_meta_5core_2014.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3443 entries, 0 to 3442
Data columns (total 17 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   main_category    3338 non-null   object 
 1   title            3443 non-null   object 
 2   average_rating   3443 non-null   float64
 3   rating_number    3443 non-null   int64  
 4   features         3443 non-null   object 
 5   description      3443 non-null   object 
 6   price            522 non-null    float64
 7   images           3443 non-null   object 
 8   videos           3443 non-null   object 
 9   store            3427 non-null   object 
 10  categories       3443 non-null   object 
 11  details          3443 non-null   object 
 12  parent_asin      3443 non-null   object 
 13  bought_together  0 non-null      float64
 14  subtitle         0 non-null      float64
 15  author           0 non-null      object 
 16  df_index         3443 non-null   int64  
dtypes: float64(4),

In [76]:
!cp "{PATH}/2014/meta_Baby.json.gz" meta_Baby.json.gz

In [77]:
!gunzip meta_Baby.json.gz

In [81]:
df_5core.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 160792 entries, 0 to 160791
Data columns (total 4 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   userID     160792 non-null  object 
 1   itemID     160792 non-null  object 
 2   rating     160792 non-null  float64
 3   timestamp  160792 non-null  int64  
dtypes: float64(1), int64(1), object(2)
memory usage: 4.9+ MB


In [84]:
asin5core = set(df_5core["itemID"].unique())

In [85]:
len(asin5core)

7050

In [96]:
!cat meta_Baby.json | head

{'asin': '0188399313', 'categories': [['Baby']], 'description': 'Wee-Go Glass baby bottles by LifeFactory (Babylife) are designed to grow with your child. The included clear cover can also serve as an easy to hold cup. Twist on the solid cap (sold separately) and use your bottles for storing juice or snacks. Perfect for a lunchbox or traveling. The bright colored silicone sleeve (patent pending) helps to protect the bottle from breakage and provides a great gripping surface and tactile experience during feeding. The bottle and sleeve can be boiled or put in the dishwasher together. They can also go in the freezer, making breast milk storage simple.', 'title': 'Lifefactory 4oz BPA Free Glass Baby Bottles - 4-pack-raspberry and Lilac', 'price': 69.99, 'imUrl': 'http://ecx.images-amazon.com/images/I/41SwthpdD9L._SX300_.jpg', 'brand': 'Lifefactory', 'related': {'also_bought': ['B002SG7K7A', 'B003CJSXW8', 'B004PW4186', 'B002O3JH9Q', 'B002O3NLIO', 'B004HGSU28'], 'also_viewed': ['B003CJSXW8',

In [104]:
import ast
import os
import json

def fast_normalize(input_path, output_path, chunk_size=1000):
    print(f"Bắt đầu chuẩn hóa: {input_path}")

    with open(input_path, 'r', encoding='utf-8') as f_in, \
         open(output_path, 'w', encoding='utf-8') as f_out:

        buffer = []
        for line in f_in:
            line = line.strip()
            if not line: continue

            try:
                # ast.literal_eval nhanh và chuẩn hơn eval cho bài toán này
                data_dict = ast.literal_eval(line)
                buffer.append(json.dumps(data_dict) + '\n')

                # Khi đủ 1000 dòng thì ghi xuống ổ cứng 1 lần
                if len(buffer) >= chunk_size:
                    f_out.writelines(buffer)
                    buffer = []
            except:
                continue

        # Ghi nốt phần còn lại trong buffer
        if buffer:
            f_out.writelines(buffer)

    print(f"Hoàn thành! Đã lưu tại: {output_path}")
    print(f"Kích thước file mới: {os.path.getsize(output_path) / 1024 / 1024:.2f} MB")

In [105]:
fast_normalize("/content/meta_Baby.json", "/content/meta_Baby.jsonl")

Bắt đầu chuẩn hóa: /content/meta_Baby.json
Hoàn thành! Đã lưu tại: /content/meta_Baby.jsonl
Kích thước file mới: 105.82 MB


In [106]:
df_meta_2014 = pd.read_json("meta_Baby.jsonl", lines=True)

In [107]:
df_meta_2014.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 71317 entries, 0 to 71316
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   asin         71317 non-null  object 
 1   categories   71317 non-null  object 
 2   description  65642 non-null  object 
 3   title        71241 non-null  object 
 4   price        57741 non-null  float64
 5   imUrl        71243 non-null  object 
 6   brand        27858 non-null  object 
 7   related      58721 non-null  object 
 8   salesRank    36 non-null     object 
dtypes: float64(1), object(8)
memory usage: 4.9+ MB


In [108]:
df_meta_5core_2014 = df_meta_2014[df_meta_2014["asin"].isin(asin5core)].copy()

In [109]:
df_meta_5core_2014.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7050 entries, 7 to 71289
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   asin         7050 non-null   object 
 1   categories   7050 non-null   object 
 2   description  6956 non-null   object 
 3   title        7037 non-null   object 
 4   price        6883 non-null   float64
 5   imUrl        7037 non-null   object 
 6   brand        4926 non-null   object 
 7   related      6946 non-null   object 
 8   salesRank    4 non-null      object 
dtypes: float64(1), object(8)
memory usage: 550.8+ KB


In [110]:
df_meta_5core_2014.to_parquet("df_meta_5core_2014.parquet")

In [111]:
!cp df_meta_5core_2014.parquet "{PATH}/2014/df_meta_5core_2014.parquet"

In [116]:
df_meta_5core_2014

,asin,categories,description,title,price,imUrl,brand,related,salesRank
7,097293751X,[[Baby]],Easily keep track of your baby's or child's da...,"Baby Tracker&reg; - Daily Childcare Journal, S...",17.00,http://ecx.images-amazon.com/images/I/41Bb6wf%...,Time Too,"{'also_bought': ['9729375011', 'B004FN1AE8', '...",NaN
108,9729375011,[[Baby]],This is version of the award-winningBaby Track...,Newborn Baby Tracker&reg; - Round the Clock Ch...,15.95,http://ecx.images-amazon.com/images/I/51r3BLpL...,NaN,"{'also_bought': ['B000V5KPZ4', 'B001F8TLLU', '...",NaN
134,B00000IZQI,[[Baby]],This colorful car collection develops motor sk...,Fisher Price Nesting Action Vehicles,8.37,http://ecx.images-amazon.com/images/I/51E83QCC...,NaN,"{'also_bought': ['B0042D69W4', 'B00428LIZM', '...",NaN
144,B00000J3LL,[[Baby]],This darling cloth book offers hands-on experi...,"My Quiet Book, Fabric Activity Book for Children",27.00,http://ecx.images-amazon.com/images/I/51GoNXhB...,NaN,"{'also_bought': ['B00000J3LC', 'B0043G4JOA', '...",NaN
163,B00002JV9S,[[Baby]],"In a relatively new concept in teething, The F...",The First Years Massaging Action Teether,8.84,http://ecx.images-amazon.com/images/I/41gVp98n...,The First Years,"{'also_bought': ['B0013FCBJO', 'B0019QCGVK', '...",NaN
...,...,...,...,...,...,...,...,...,...
70839,B00JEV5UI8,[[Baby]],,VTech Communications Safe and Sound Digital Au...,19.99,http://ecx.images-amazon.com/images/I/41KvPcQ-...,NaN,"{'also_bought': ['B007INU6EC', 'B00BRHSD90', '...",NaN
70927,B00JLI73ZM,[[Baby]],Parents dont vibrate like bouncy seats or swin...,4Moms mamaRoo Plush Infant Seat - Silver,249.00,http://ecx.images-amazon.com/images/I/41-lfZws...,4moms,"{'also_bought': ['B007ZXX03Y', 'B00BXWTVJ0', '...",NaN
70930,B00JLHWDO4,[[Baby]],Baby's weight is supported by buttocks and upp...,Merry Muscles Ergonomic Jumper Exerciser Baby ...,89.95,http://ecx.images-amazon.com/images/I/41Z5Is6e...,NaN,"{'also_bought': ['B003I1XA0U', 'B008BGNT3A', '...",NaN
71006,B00JRYRYS6,[[Baby]],NaN,"Graco Argos 65 3-in-1 Harness Booster, Link",199.99,http://ecx.images-amazon.com/images/I/41vnR97a...,Graco,"{'also_bought': ['B00H8MSKAU', 'B00HG699OO', '...",NaN


In [121]:
df_meta_5core_2014.columns

Index(['asin', 'categories', 'description', 'title', 'price', 'imUrl', 'brand',
       'related', 'salesRank'],
      dtype='object')

In [119]:
df_meta_5core_2014["imUrl"]

,imUrl
7,http://ecx.images-amazon.com/images/I/41Bb6wf%...
108,http://ecx.images-amazon.com/images/I/51r3BLpL...
134,http://ecx.images-amazon.com/images/I/51E83QCC...
144,http://ecx.images-amazon.com/images/I/51GoNXhB...
163,http://ecx.images-amazon.com/images/I/41gVp98n...
...,...
70839,http://ecx.images-amazon.com/images/I/41KvPcQ-...
70927,http://ecx.images-amazon.com/images/I/41-lfZws...
70930,http://ecx.images-amazon.com/images/I/41Z5Is6e...
71006,http://ecx.images-amazon.com/images/I/41vnR97a...


In [120]:
df_filtered_meta_5core_2014["images"]

,images
0,[{'hi_res': 'https://m.media-amazon.com/images...
1,"[{'hi_res': None, 'large': 'https://m.media-am..."
2,"[{'hi_res': None, 'large': 'https://m.media-am..."
3,[{'hi_res': 'https://m.media-amazon.com/images...
4,[{'hi_res': 'https://m.media-amazon.com/images...
...,...
3438,"[{'hi_res': None, 'large': 'https://m.media-am..."
3439,[{'hi_res': 'https://m.media-amazon.com/images...
3440,"[{'hi_res': None, 'large': 'https://m.media-am..."
3441,[{'hi_res': 'https://m.media-amazon.com/images...


In [122]:
df_filtered_found = df_meta_5core_2014[df_meta_5core_2014["asin"].isin(set(df_filtered_meta_5core_2014["parent_asin"]))]

In [123]:
df_filtered_found.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3443 entries, 7 to 71289
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   asin         3443 non-null   object 
 1   categories   3443 non-null   object 
 2   description  3400 non-null   object 
 3   title        3437 non-null   object 
 4   price        3356 non-null   float64
 5   imUrl        3437 non-null   object 
 6   brand        2335 non-null   object 
 7   related      3394 non-null   object 
 8   salesRank    0 non-null      object 
dtypes: float64(1), object(8)
memory usage: 269.0+ KB


## Read jonl

In [16]:
reviews_chunks = pd.read_json(os.path.join(PATH, "2023/Baby_Products.jsonl"), lines=True, chunksize=100_000)

In [17]:
!cp "{PATH}/2023/Baby_Products.jsonl" "Baby_Products.jsonl"

In [31]:
import pandas as pd
import pyarrow.parquet as pq
import pyarrow as pa
import os
import gc

def convert_jsonl_to_parquet_snappy(jsonl_path, parquet_path, chunk_size=100_000):
    """
    Chuyển đổi file JSONL sang Parquet với nén Snappy để tránh crash RAM.
    """
    # Khởi tạo reader theo từng cụm (chunk)
    reviews_chunks = pd.read_json(jsonl_path, lines=True, chunksize=chunk_size)

    writer = None

    print(f"Bắt đầu chuyển đổi: {jsonl_path} -> {parquet_path}")

    try:
        for i, chunk in enumerate(reviews_chunks):
            # 1. Ép kiểu object (thường là string/mixed) sang string thuần
            # Việc này giúp tránh lỗi không đồng nhất kiểu dữ liệu (Schema mismatch) trong Parquet
            for col in chunk.select_dtypes(include=['object']).columns:
                chunk[col] = chunk[col].astype(str)

            # 2. Chuyển sang bảng Arrow
            table = pa.Table.from_pandas(chunk)

            # 3. Khởi tạo writer ở vòng lặp đầu tiên với nén Snappy
            if writer is None:
                writer = pq.ParquetWriter(parquet_path, table.schema, compression='snappy')

            # 4. Ghi chunk hiện tại vào file
            writer.write_table(table)

            # 5. Dọn dẹp bộ nhớ ngay lập tức
            del chunk
            del table
            gc.collect()

            print(f"--- Đã xử lý xong chunk {i+1} ({((i+1)*chunk_size):,}) dòng ---")

    except Exception as e:
        print(f"Lỗi trong quá trình chuyển đổi: {e}")

    finally:
        # Đảm bảo đóng file dù có lỗi xảy ra hay không
        if writer:
            writer.close()
            print(f"\n[XONG] File Parquet Snappy đã được lưu tại: {parquet_path}")

# --- Cách sử dụng ---
# convert_jsonl_to_parquet_snappy("Baby_Products.jsonl", "Baby_Products.parquet")

In [30]:
!ls -lh

total 4.3G
-rw------- 1 root root 2.8G Mar 27 08:04 Baby_Products.jsonl
-rw-r--r-- 1 root root 1.1G Mar 27 08:06 Baby_Products.parquet
-rw------- 1 root root 167M Mar 27 07:54 df_meta_2023.gzip.parquet
-rw-r--r-- 1 root root 265M Mar 27 08:11 df_meta_2023.parquet
drwx------ 5 root root 4.0K Mar 27 07:51 drive
drwxr-xr-x 1 root root 4.0K Mar 23 13:29 sample_data


In [20]:
!cp Baby_Products.parquet "{PATH}/2023/Baby_Products.parquet"

In [29]:
!cp df_meta_2023.parquet "{PATH}/2023/df_meta_2023.parquet"